<a href="https://colab.research.google.com/github/geqqlong-bit/ComfyUI_H3_Wan/blob/main/ComfyUI_H3_WanNSFW_1080p.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 ComfyUI 1080p+ 视频生成工作站（MiniMax-H3 音画同出 + 社区 NSFW 版 Wan 2.2 + 自定义 LoRA）

本笔记专为 **Google Colab (`L4` / `A100` + 高 RAM)** 环境深度调优，开箱集成：
- **MiniMax-H3 (33B)**：支持文生音视频 / 首尾帧图生音视频，搭载 `FP8_scaled` 主模型 + 全精度双 VAE + **3D 潜空间 1080p/2K 放大器（保留原生立体声效）**。
- **Wan 2.2 社区 NSFW/Spicy 无限制版**：搭载无审查编码器 `NSFW-Wan-UMT5-XXL` + `Wan2.2-Remix NSFW`（I2V v3.0 / T2V v2.0）或 `Phr00t Rapid-Mega-AIO NSFW`，完美兼容社区高/低噪自定义 LoRA。
- **通用真 1080p/2K 超分管线**：内置 `SeedVR2 Video Upscaler` 单步扩散视频超分与 `RIFE` 视频丝滑补帧。
- **持久化云端资产**：自动将生成的视频、工作流画布及**私人 LoRA 文件夹**映射至 Google Drive，断联不丢文件、无需重复下载 LoRA。

> ⚠️ **运行前必读**：
> 1. **算力建议**：日常玩 `Wan 2.2 NSFW + LoRA + 1080p` 选 **L4 GPU（高 RAM）** 性价比最高；跑 `MiniMax-H3` 建议切 **A100 GPU**。
> 2. **协议授权**：若勾选下载 MiniMax-H3，请先登录你的 Hugging Face 账号并访问 [Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3) 点击同意协议（Agree/Accept），否则 API 会报 `401/403` 权限错误。

In [ ]:
#@title 1. 环境自检与 Google Drive 持久化挂载 { display-mode: "form" }
#@markdown 勾选后会自动将 `output`（导出视频）、`workflows`（工作流画布）和 `loras`（私人持久化 LoRA 库）关联至 Google Drive：
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_FOLDER_NAME = "ComfyUI_Video_Studio" #@param {type:"string"}

import os
import shutil
import psutil
import torch

print("=" * 55)
print("🖥️ 系统硬件检测报告")
print("=" * 55)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU 型号: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    if vram_gb < 20:
        print("⚠️ 警告: 当前显存小于 20GB (可能是 T4)。跑 H3 / Wan 14B 1080p 极易爆显存，强烈建议切换至 L4 或 A100！")
else:
    raise RuntimeError("❌ 未检测到 GPU！请在菜单栏选择：代码执行程序 -> 更改运行时类型 -> 选择 L4 或 A100 GPU。")

ram_gb = psutil.virtual_memory().total / (1024**3)
disk_free_gb = shutil.disk_usage("/content").free / (1024**3)
print(f"✅ 系统内存 (RAM): {ram_gb:.1f} GB")
print(f"✅ 可用磁盘空间: {disk_free_gb:.1f} GB")
if ram_gb < 40:
    print("⚠️ 提示: 系统内存低于 40GB，建议在『更改运行时类型』中开启『高 RAM (High-RAM)』。")
if disk_free_gb < 90:
    print("⚠️ 提示: 同时下载全部模型约需 80~100GB 空间，请按可用磁盘空间合理勾选模型。")

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = f"/content/drive/MyDrive/{DRIVE_FOLDER_NAME}"
    os.makedirs(f"{DRIVE_BASE}/output", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/workflows", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/loras", exist_ok=True)
    print(f"✅ Google Drive 已挂载，持久化目录就绪: {DRIVE_BASE}")

In [ ]:
#@title 2. 绑定 Hugging Face API Token { display-mode: "form" }
#@markdown **方式一（推荐）**：点击 Colab 左侧边栏的 🔑（密钥/Secrets）图标，添加名称为 `HF_TOKEN` 的密钥并开启访问开关，下方留空即可。<br>
#@markdown **方式二**：直接在下方粘贴你的 Hugging Face Access Token（`hf_...`）。
HF_API_TOKEN = "" #@param {type:"string"}

import os
import subprocess

subprocess.run(["pip", "install", "-q", "-U", "huggingface_hub[cli,hf_transfer]"], check=True)

from huggingface_hub import login, whoami

token_to_use = HF_API_TOKEN.strip()
if not token_to_use:
    try:
        from google.colab import userdata
        token_to_use = userdata.get('HF_TOKEN')
        print("🔐 已从 Colab Secrets 安全读取 HF_TOKEN")
    except Exception:
        pass

if not token_to_use:
    raise ValueError("❌ 未检测到 Hugging Face Token！请在表单中输入，或在左侧 🔑 Secrets 中配置 HF_TOKEN。")

os.environ["HF_TOKEN"] = token_to_use
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

login(token=token_to_use, add_to_git_credential=True)
user_info = whoami(token=token_to_use)
print(f"✅ Hugging Face 认证成功！当前绑定账号: {user_info['name']}")
print("📌 提醒: 若需下载 MiniMax-H3，请确认该账号已在 https://huggingface.co/Comfy-Org/MiniMax-H3 点击 Agree。")

In [ ]:
#@title 3. 部署 ComfyUI 本体 + 10 大主流插件 + 挂载云端私人 LoRA 库 { display-mode: "form" }
INSTALL_SAGE_ATTENTION = True #@param {type:"boolean"}

import os
import shutil
import subprocess

COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("🚀 正在克隆最新版 ComfyUI (含原生 MiniMax-H3 / Wan 节点与官方工作流模板)...")
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_DIR], check=True)
else:
    print("🔄 正在更新 ComfyUI 至最新版本...")
    subprocess.run(["git", "-C", COMFY_DIR, "pull"], check=False)

print("📦 正在安装 ComfyUI 核心依赖与工作流模板包...")
subprocess.run(["pip", "install", "-q", "-r", f"{COMFY_DIR}/requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-U", "comfyui-frontend-package", "comfyui-workflow-templates", "gguf", "accelerate"], check=True)

if INSTALL_SAGE_ATTENTION:
    print("⚡ 正在安装 SageAttention (大幅加速视频注意力推理)...")
    res = subprocess.run(["pip", "install", "-q", "sageattention"], check=False)
    if res.returncode != 0:
        print("⚠️ SageAttention 编译跳过，将自动使用 PyTorch SDPA 原生加速。")

CUSTOM_NODES = {
    "ComfyUI-Manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "ComfyUI-Custom-Scripts": "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",
    "ComfyUI-KJNodes": "https://github.com/kijai/ComfyUI-KJNodes.git",
    "ComfyUI-VideoHelperSuite": "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "ComfyUI-WanVideoWrapper": "https://github.com/kijai/ComfyUI-WanVideoWrapper.git",
    "ComfyUI-GGUF": "https://github.com/city96/ComfyUI-GGUF.git",
    "ComfyUI-SeedVR2_VideoUpscaler": "https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git",
    "ComfyUI-Frame-Interpolation": "https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git"
}

nodes_dir = os.path.join(COMFY_DIR, "custom_nodes")
print("🧩 正在安装 10 大主流插件与画布效率脚本...")
for name, url in CUSTOM_NODES.items():
    target = os.path.join(nodes_dir, name)
    if not os.path.exists(target):
        print(f"  -> 安装 {name} ...")
        subprocess.run(["git", "clone", "--depth", "1", url, target], check=True)
        req_file = os.path.join(target, "requirements.txt")
        if os.path.exists(req_file):
            subprocess.run(["pip", "install", "-q", "-r", req_file], check=False)

if globals().get("MOUNT_GOOGLE_DRIVE", False) and os.path.exists("/content/drive/MyDrive"):
    folder_name = globals().get("DRIVE_FOLDER_NAME", "ComfyUI_Video_Studio")
    DRIVE_BASE = f"/content/drive/MyDrive/{folder_name}"
    os.makedirs(f"{DRIVE_BASE}/output", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/workflows", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/loras", exist_ok=True)

    local_out = os.path.join(COMFY_DIR, "output")
    if os.path.exists(local_out) and not os.path.islink(local_out):
        shutil.rmtree(local_out)
    if not os.path.exists(local_out):
        os.symlink(f"{DRIVE_BASE}/output", local_out)

    user_wf_dir = os.path.join(COMFY_DIR, "user", "default", "workflows")
    os.makedirs(os.path.dirname(user_wf_dir), exist_ok=True)
    if os.path.exists(user_wf_dir) and not os.path.islink(user_wf_dir):
        shutil.rmtree(user_wf_dir)
    if not os.path.exists(user_wf_dir):
        os.symlink(f"{DRIVE_BASE}/workflows", user_wf_dir)

    extra_yaml_path = os.path.join(COMFY_DIR, "extra_model_paths.yaml")
    with open(extra_yaml_path, "w", encoding="utf-8") as f:
        f.write(f"colab_drive:\n  base_path: {DRIVE_BASE}\n  loras: loras\n")
    print(f"✅ 已将视频输出、画布工作流及私人 LoRA 库 ({DRIVE_BASE}/loras) 自动关联至 Google Drive！")

print("🎉 ComfyUI 本体与全套插件部署完成！")

In [ ]:
#@title 4. 下载 MiniMax-H3 + 社区 NSFW/Spicy 版 Wan 2.2 + 1080p 超分模型 { display-mode: "form" }
#@markdown ### 🎬 MiniMax-H3 视频模型选项（L4/A100 黄金混搭：FP8主模型 + 全精度双VAE）
DOWNLOAD_MINIMAX_H3_FL2VA = True #@param {type:"boolean"}
DOWNLOAD_MINIMAX_H3_REF2VA = False #@param {type:"boolean"}
H3_PRECISION = "fp8_scaled (Colab兼容性最佳)" #@param ["fp8_scaled (Colab兼容性最佳)", "int8_convrot (需CUDA13环境)"]
DOWNLOAD_H3_TURBO_LORA = True #@param {type:"boolean"}
DOWNLOAD_H3_LATENT_UPSCALER = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🔥 Wan 2.2 社区 NSFW / Spicy 无限制版（内置无审查编码器，支持自由叠加 LoRA）
WAN_NSFW_EDITION = "Wan2.2-Remix NSFW I2V v3.0 (图生视频旗舰 FP8 - 推荐)" #@param ["Wan2.2-Remix NSFW I2V v3.0 (图生视频旗舰 FP8 - 推荐)", "Wan2.2-Remix NSFW T2V v2.0 (文生视频旗舰 FP8)", "Wan2.2-Remix NSFW I2V+T2V 双全套 (约55GB)", "Phr00t Rapid-Mega-AIO NSFW v12.2 (4步极速图文合一单文件)", "仅下载无审查CLIP+VAE", "不下载"]

#@markdown ---
#@markdown ### 🔍 通用 1080p / 2K 视频超分增强（SeedVR2 单步扩散超分）
DOWNLOAD_SEEDVR2_1080P_UPSCALER = True #@param {type:"boolean"}

import os
import shutil
from huggingface_hub import hf_hub_download

MODELS_DIR = "/content/ComfyUI/models"
STAGING_DIR = "/content/hf_temp_staging"
hf_token = os.environ.get("HF_TOKEN")

def fetch_hf_model(repo_id: str, repo_filepath: str, target_subdir: str, rename_to: str = None):
    target_folder = os.path.join(MODELS_DIR, target_subdir)
    os.makedirs(target_folder, exist_ok=True)
    filename = rename_to if rename_to else os.path.basename(repo_filepath)
    dest_path = os.path.join(target_folder, filename)

    if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024 * 1024:
        print(f"⏭️ 已存在，跳过: {target_subdir}/{filename}")
        return dest_path

    print(f"⬇️ 正在拉取 [{repo_id}] -> {filename} ...")
    os.makedirs(STAGING_DIR, exist_ok=True)
    downloaded_path = hf_hub_download(
        repo_id=repo_id,
        filename=repo_filepath,
        local_dir=STAGING_DIR,
        token=hf_token
    )
    shutil.move(downloaded_path, dest_path)
    shutil.rmtree(STAGING_DIR, ignore_errors=True)
    print(f"✅ 就位: {dest_path}")
    return dest_path

# ================= 1. 下载 MiniMax-H3 系列 =================
h3_repo = "Comfy-Org/MiniMax-H3"
prec_suffix = "fp8_scaled" if "fp8_scaled" in H3_PRECISION else "int8_convrot"

if DOWNLOAD_MINIMAX_H3_FL2VA or DOWNLOAD_MINIMAX_H3_REF2VA:
    fetch_hf_model(h3_repo, "text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders")
    fetch_hf_model(h3_repo, "vae/minimax_h3_video_vae_fp16.safetensors", "vae")
    fetch_hf_model(h3_repo, "vae/minimax_h3_audio_vae_fp32.safetensors", "vae")

if DOWNLOAD_MINIMAX_H3_FL2VA:
    fetch_hf_model(h3_repo, f"diffusion_models/minimax_h3_fl2va_pruned_{prec_suffix}.safetensors", "diffusion_models")
    if DOWNLOAD_H3_TURBO_LORA:
        fetch_hf_model(h3_repo, "loras/minimax_h3_fl2v_turbo_4step_v1.0_768p_comfyui_bf16.safetensors", "loras")

if DOWNLOAD_MINIMAX_H3_REF2VA:
    fetch_hf_model(h3_repo, f"diffusion_models/minimax_h3_ref2va_pruned_{prec_suffix}.safetensors", "diffusion_models")
    if DOWNLOAD_H3_TURBO_LORA:
        fetch_hf_model(h3_repo, "loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors", "loras")

if DOWNLOAD_H3_LATENT_UPSCALER:
    try:
        fetch_hf_model("LBH-123-AI/Minimax_h3_latent_Upscaler", "minimax_h3_latent_upscaler_3d_fp16.safetensors", "latent_upscale_models")
    except Exception:
        fetch_hf_model("LBH-123-AI/Minimax_h3_latent_Upscaler", "minimax_h3_latent_upscaler_3d_conv_v1/minimax_h3_latent_upscaler_3d_conv_v1_fp16.safetensors", "latent_upscale_models")

# ================= 2. 下载社区 NSFW/Spicy 版 Wan 2.2 =================
if WAN_NSFW_EDITION != "不下载":
    fetch_hf_model("NSFW-API/NSFW-Wan-UMT5-XXL", "nsfw_wan_umt5-xxl_fp8_scaled.safetensors", "text_encoders")
    fetch_hf_model("Comfy-Org/Wan_2.2_ComfyUI_Repackaged", "split_files/vae/wan_2.1_vae.safetensors", "vae")

    remix_repo = "FX-FeiHou/wan2.2-Remix"
    if "I2V v3.0" in WAN_NSFW_EDITION or "双全套" in WAN_NSFW_EDITION:
        fetch_hf_model(remix_repo, "NSFW/Wan2.2_Remix_NSFW_i2v_14b_high_lighting_fp8_e4m3fn_v3.0.safetensors", "diffusion_models")
        fetch_hf_model(remix_repo, "NSFW/Wan2.2_Remix_NSFW_i2v_14b_low_lighting_fp8_e4m3fn_v3.0.safetensors", "diffusion_models")

    if "T2V v2.0" in WAN_NSFW_EDITION or "双全套" in WAN_NSFW_EDITION:
        fetch_hf_model(remix_repo, "NSFW/Wan2.2_Remix_NSFW_t2v_14b_high_lighting_v2.0.safetensors", "diffusion_models")
        fetch_hf_model(remix_repo, "NSFW/Wan2.2_Remix_NSFW_t2v_14b_low_lighting_v2.0.safetensors", "diffusion_models")

    if "Phr00t" in WAN_NSFW_EDITION:
        fetch_hf_model("Phr00t/WAN2.2-14B-Rapid-AllInOne", "Mega-v12/wan2.2-rapid-mega-aio-nsfw-v12.2.safetensors", "checkpoints")

# ================= 3. 下载 SeedVR2 1080p 视频超分模型 =================
if DOWNLOAD_SEEDVR2_1080P_UPSCALER:
    fetch_hf_model("numz/SeedVR2_comfyUI", "seedvr2_ema_3b_fp8_e4m3fn.safetensors", "SEEDVR2")
    fetch_hf_model("numz/SeedVR2_comfyUI", "ema_vae_fp16.safetensors", "SEEDVR2")

disk_left = shutil.disk_usage("/content").free / (1024**3)
print("=" * 55)
print(f"🎉 所有选中模型已全部就位！当前剩余磁盘空间: {disk_left:.1f} GB")

In [ ]:
#@title 5. 自定义 LoRA 批量导入器（支持 HuggingFace / Civitai / 直链） { display-mode: "form" }
#@markdown 下载 Civitai 受年龄限制的 NSFW LoRA 时，请填入 Civitai API Key（或在左侧 🔑 Secrets 中添加 `CIVITAI_TOKEN`）：
CIVITAI_API_TOKEN = "" #@param {type:"string"}
#@markdown 勾选后，下载的自定义 LoRA 将永久保存至你的 Google Drive `loras` 文件夹（下次开机自动加载）：
SAVE_LORA_TO_DRIVE = True #@param {type:"boolean"}
#@markdown 粘贴 LoRA 链接（多个链接用英文逗号 `,` 分隔；支持 HF 页面/下载链接、C站详情页链接或直链）：
CUSTOM_LORA_URLS = "" #@param {type:"string"}

import os
import re
import shutil
import urllib.parse
import requests
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

civitai_token = CIVITAI_API_TOKEN.strip()
if not civitai_token:
    try:
        from google.colab import userdata
        civitai_token = userdata.get('CIVITAI_TOKEN')
    except Exception:
        civitai_token = ""

hf_token = os.environ.get("HF_TOKEN", "")

if SAVE_LORA_TO_DRIVE and os.path.exists("/content/drive/MyDrive"):
    folder_name = globals().get("DRIVE_FOLDER_NAME", "ComfyUI_Video_Studio")
    LORA_SAVE_DIR = f"/content/drive/MyDrive/{folder_name}/loras"
else:
    LORA_SAVE_DIR = "/content/ComfyUI/models/loras"
os.makedirs(LORA_SAVE_DIR, exist_ok=True)

def download_custom_lora(raw_url: str):
    url = raw_url.strip()
    if not url:
        return

    hf_match = re.match(r"https?://huggingface\.co/([^/]+/[^/]+)/(?:blob|resolve)/([^/]+)/(.+)", url)
    if hf_match:
        repo_id, revision, filepath = hf_match.groups()
        filepath = urllib.parse.unquote(filepath.split("?")[0])
        fname = os.path.basename(filepath)
        dest = os.path.join(LORA_SAVE_DIR, fname)
        print(f"🤗 [HuggingFace] 正在拉取 {repo_id} -> {fname} ...")
        temp_dir = "/content/hf_lora_tmp"
        temp_dl = hf_hub_download(repo_id=repo_id, filename=filepath, revision=revision, local_dir=temp_dir, token=hf_token or None)
        shutil.move(temp_dl, dest)
        shutil.rmtree(temp_dir, ignore_errors=True)
        print(f"✅ 已保存至: {dest}")
        return

    if "civitai.com" in url:
        ver_match = re.search(r"modelVersionId=(\d+)", url) or re.search(r"/api/download/models/(\d+)", url)
        if ver_match:
            ver_id = ver_match.group(1)
            url = f"https://civitai.com/api/download/models/{ver_id}"
        elif "/models/" in url:
            model_id_match = re.search(r"/models/(\d+)", url)
            if model_id_match:
                meta = requests.get(f"https://civitai.com/api/v1/models/{model_id_match.group(1)}").json()
                ver_id = meta["modelVersions"][0]["id"]
                url = f"https://civitai.com/api/download/models/{ver_id}"

        if civitai_token and "token=" not in url:
            sep = "&" if "?" in url else "?"
            url = f"{url}{sep}token={civitai_token}"

    headers = {"User-Agent": "Mozilla/5.0"}
    if civitai_token and "civitai.com" in url:
        headers["Authorization"] = f"Bearer {civitai_token}"

    with requests.get(url, headers=headers, stream=True, allow_redirects=True) as r:
        r.raise_for_status()
        if "text/html" in r.headers.get("Content-Type", ""):
            raise RuntimeError("❌ 下载返回了登录网页而非模型文件！该 NSFW LoRA 需要鉴权，请填写 CIVITAI_API_TOKEN。")

        cd = r.headers.get("Content-Disposition", "")
        fname_match = re.findall(r"""filename\*?=(?:UTF-8'')?["']?([^"';\r\n]+)["']?""", cd, flags=re.IGNORECASE)
        if fname_match:
            fname = urllib.parse.unquote(fname_match[0])
        else:
            fname = os.path.basename(urllib.parse.urlparse(r.url).path) or "custom_lora.safetensors"
            if not fname.endswith((".safetensors", ".pt", ".ckpt")):
                fname += ".safetensors"

        dest = os.path.join(LORA_SAVE_DIR, fname)
        total = int(r.headers.get("content-length", 0))
        print(f"⬇️ 正在下载 LoRA: {fname} ({total / (1024**2):.1f} MB) ...")
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
        print(f"✅ 已保存至: {dest}")

urls = [u.strip() for u in CUSTOM_LORA_URLS.split(",") if u.strip()]
if not urls:
    print(f"ℹ️ 当前未填写下载链接。你也可以直接把本地 LoRA 文件拖入云盘目录: {LORA_SAVE_DIR}")
else:
    for u in urls:
        download_custom_lora(u)
    print("🎉 自定义 LoRA 导入完毕！若 ComfyUI 已运行，在网页画布按键盘『R』键即可刷新节点列表。")

In [ ]:
#@title 6. 一键启动 ComfyUI（Cloudflare 极速隧道 + Colab 原生代理双通道） { display-mode: "form" }
EXTRA_LAUNCH_ARGS = "--enable-manager" #@param {type:"string"}

import os
import re
import time
import shlex
import threading
import subprocess
import urllib.request
from google.colab import output

cloudflared_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cloudflared_bin):
    print("🌐 正在配置 Cloudflare 极速公网隧道客户端...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared_bin
    )
    os.chmod(cloudflared_bin, 0o755)

def start_tunnels(port=8188):
    while True:
        time.sleep(2)
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}")
            break
        except Exception:
            pass

    print("\n" + "=" * 65)
    print("✅ ComfyUI 服务已就绪！请点击下方任一链接打开画布界面：")
    print("=" * 65)

    try:
        colab_url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
        print(f"🔗 [备用通道 - Colab 原生代理]: {colab_url}")
    except Exception:
        pass

    p = subprocess.Popen(
        [cloudflared_bin, "tunnel", "--url", f"http://127.0.0.1:{port}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    for line in p.stderr:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print(f"🚀 [推荐首选 - Cloudflare 极速链接]: {match.group(0)}\n")
            print("=" * 65, flush=True)
            break

threading.Thread(target=start_tunnels, daemon=True).start()

os.chdir("/content/ComfyUI")
os.environ["PYTHONUNBUFFERED"] = "1"
cmd = ["python", "-u", "main.py", "--listen", "0.0.0.0", "--port", "8188"] + shlex.split(EXTRA_LAUNCH_ARGS)

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    print("\n🛑 正在停止 ComfyUI 服务...")
    proc.terminate()
    proc.wait()

---
### 📖 画布快速上手与 1080p + 自定义 LoRA 连线速查

1. **一键加载官方模板**：进入界面后点击左上角 **`Workflow` -> `Browse Templates` -> `Video`**，可直接打开 **MiniMax-H3** 或 **Wan 2.2** 标准模板，将模型加载器（Loader）切换为已下载的权重即可。
2. **Wan 2.2 NSFW 挂载自定义 LoRA 方法**：
   - `LoadCLIP` 选择无审查编码器 **`nsfw_wan_umt5-xxl_fp8_scaled.safetensors`**。
   - 在 High Noise 和 Low Noise 两个 `Load Diffusion Model` 节点后，各串联一个 **`Power Lora Loader (rgthree)`** 或 **`LoraLoaderModelOnly`**：若社区 LoRA 区分 `_HIGH` / `_LOW` 则对应挂载；若是通用单文件 LoRA 可两边同挂（建议权重 **`0.5 ~ 0.8`**）。
   - 配合 `Wan2.2-Remix` 内置的 Lightning 加速，`CFG` 设为 **`1.0`**，总步数 **`10 ~ 12` 步**（如 High 跑 `0->5`，Low 跑 `5->10`）即可极速出片。
3. **双管线直出真 1080p / 2K 视频**：
   - **MiniMax-H3 原声 1080p/2K**：在采样器之后、`H3 VAE Decode` 之前接入 **`MiniMax H3 Latent Upscale With Model`**（加载 `minimax_h3_latent_upscaler_3d_fp16`），分辨率设为 `1920×1080` 或 `2560×1440`，可完整保留原生立体声对白与音效。
   - **Wan 2.2 / 通用真 1080p 超分**：先按 `1280×720` 快速生成视频帧，在 `VAE Decode` 后接入 **`SeedVR2 Video Upscaler`**（加载 `seedvr2_ema_3b_fp8`，短边 `resolution` 设为 **`1080`**），最后连接 `VHS Video Combine` 导出 MP4。